In [ ]:
# ============================================
# Cell 1: Install dependencies and imports
# ============================================
!pip install -q timm matplotlib torch-ema

import os
import numpy as np
import pandas as pd
import pydicom
from PIL import Image
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
import torch.nn.functional as F
import timm
from tqdm import tqdm
from sklearn.model_selection import train_test_split
from sklearn.metrics import roc_auc_score
from torch_ema import ExponentialMovingAverage
import warnings
import time
import pickle
import matplotlib.pyplot as plt
warnings.filterwarnings('ignore')

print("Libraries imported successfully!")


# ============================================
# Cell 2: Enhanced Configuration
# ============================================
class CFG:
    DATA_DIR = '/kaggle/input/rsna-intracranial-aneurysm-detection/series'
    TRAIN_CSV = '/kaggle/input/rsna-intracranial-aneurysm-detection/train.csv'
    
    CACHE_DIR = './preprocessed_cache'
    USE_CACHE = True
    
    BACKBONE = 'vit_small_patch16_224'
    NUM_SLICES = 20
    IMAGE_SIZE = 224
    NUM_CLASSES = 14
    
    EPOCHS = 50
    BATCH_SIZE = 8
    LR = 2e-5
    NUM_WORKERS = 0
    
    # Single seed training
    SEEDS = [42]
    
    USE_SUBSET = None
    PATIENCE = 12
    
    EMBED_DIM = 384
    NUM_HEADS = 8
    TRANSFORMER_LAYERS = 4
    DROPOUT = 0.5
    
    GRADIENT_ACCUMULATION = 2
    LABEL_SMOOTHING = 0.1
    USE_EMA = True
    
    DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'

os.makedirs(CFG.CACHE_DIR, exist_ok=True)
print(f"Device: {CFG.DEVICE}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
print(f"Training with seed: {CFG.SEEDS}")


# ============================================
# Cell 3: Focal Loss
# ============================================
class FocalLoss(nn.Module):
    def __init__(self, alpha=0.25, gamma=2.0):
        super().__init__()
        self.alpha = alpha
        self.gamma = gamma
    
    def forward(self, inputs, targets):
        BCE = F.binary_cross_entropy_with_logits(inputs, targets, reduction='none')
        pt = torch.exp(-BCE)
        focal_loss = self.alpha * (1 - pt) ** self.gamma * BCE
        return focal_loss.mean()

print("Focal Loss defined!")


# ============================================
# Cell 4: Enhanced Dataset
# ============================================
class RSNADataset(Dataset):
    def __init__(self, df, data_dir, num_slices=20, image_size=224, augment=False, 
                 cache_dir=None, label_smoothing=0.0):
        self.df = df.reset_index(drop=True)
        self.data_dir = data_dir
        self.num_slices = num_slices
        self.image_size = image_size
        self.augment = augment
        self.cache_dir = cache_dir
        self.label_smoothing = label_smoothing
        
        self.label_cols = [
            'Left Infraclinoid Internal Carotid Artery',
            'Right Infraclinoid Internal Carotid Artery',
            'Left Supraclinoid Internal Carotid Artery',
            'Right Supraclinoid Internal Carotid Artery',
            'Left Middle Cerebral Artery',
            'Right Middle Cerebral Artery',
            'Anterior Communicating Artery',
            'Left Anterior Cerebral Artery',
            'Right Anterior Cerebral Artery',
            'Left Posterior Communicating Artery',
            'Right Posterior Communicating Artery',
            'Basilar Tip',
            'Other Posterior Circulation',
            'Aneurysm Present',
        ]
        
        if self.cache_dir and CFG.USE_CACHE:
            self.preprocess_and_cache()
    
    def get_cache_path(self, series_id):
        return os.path.join(self.cache_dir, f"{series_id}_s{self.num_slices}.npy")
    
    def preprocess_and_cache(self):
        print(f"\nPreprocessing {len(self.df)} samples with {self.num_slices} slices...")
        cache_exists, cache_missing = 0, 0
        
        for idx in tqdm(range(len(self.df)), desc="Caching"):
            series_id = self.df.iloc[idx]['SeriesInstanceUID']
            cache_path = self.get_cache_path(series_id)
            
            if os.path.exists(cache_path):
                cache_exists += 1
            else:
                volume = self.load_dicom_series(series_id)
                np.save(cache_path, volume)
                cache_missing += 1
        
        print(f"Cache - Existing: {cache_exists}, Created: {cache_missing}")
    
    def load_dicom_series(self, series_id):
        try:
            series_path = os.path.join(self.data_dir, series_id)
            if not os.path.exists(series_path):
                return np.zeros((self.num_slices, self.image_size, self.image_size), dtype=np.float32)
            
            dcm_files = []
            for root, _, files in os.walk(series_path):
                for f in files:
                    if f.endswith('.dcm'):
                        dcm_files.append(os.path.join(root, f))
            
            if not dcm_files:
                return np.zeros((self.num_slices, self.image_size, self.image_size), dtype=np.float32)
            
            if len(dcm_files) > self.num_slices * 3:
                dcm_files = sorted(dcm_files)
                step = max(1, len(dcm_files) // (self.num_slices * 2))
                dcm_files = dcm_files[::step][:(self.num_slices * 2)]
            
            slices = []
            for f in dcm_files:
                try:
                    ds = pydicom.dcmread(f, force=True)
                    instance = int(getattr(ds, 'InstanceNumber', 0))
                    pixels = ds.pixel_array
                    
                    if len(pixels.shape) != 2:
                        continue
                    
                    pixels = pixels.astype(np.float32)
                    slope = float(getattr(ds, 'RescaleSlope', 1))
                    intercept = float(getattr(ds, 'RescaleIntercept', 0))
                    pixels = pixels * slope + intercept
                    
                    slices.append((instance, pixels))
                except:
                    continue
            
            if not slices:
                return np.zeros((self.num_slices, self.image_size, self.image_size), dtype=np.float32)
            
            slices.sort(key=lambda x: x[0])
            volume = np.stack([s[1] for s in slices], axis=0)
            
            n = len(volume)
            if n > self.num_slices:
                idx = np.linspace(0, n-1, self.num_slices, dtype=int)
                volume = volume[idx]
            elif n < self.num_slices:
                pad = self.num_slices - n
                volume = np.pad(volume, ((0, pad), (0, 0), (0, 0)), mode='edge')
            
            vmin, vmax = volume.min(), volume.max()
            if vmax > vmin:
                volume = (volume - vmin) / (vmax - vmin)
            else:
                volume = np.zeros_like(volume)
            
            resized = []
            for i in range(self.num_slices):
                img = Image.fromarray((volume[i] * 255).astype(np.uint8))
                img = img.resize((self.image_size, self.image_size), Image.BILINEAR)
                resized.append(np.array(img) / 255.0)
            
            volume = np.stack(resized, axis=0).astype(np.float32)
            
            mean = volume.mean()
            std = volume.std()
            if std > 1e-6:
                volume = (volume - mean) / std
            
            return volume
        except:
            return np.zeros((self.num_slices, self.image_size, self.image_size), dtype=np.float32)
    
    def augment_slice(self, slice_img):
        if np.random.rand() > 0.5:
            slice_img = np.fliplr(slice_img)
        
        if np.random.rand() > 0.5:
            angle = np.random.uniform(-10, 10)
            img_pil = Image.fromarray(((slice_img - slice_img.min()) / 
                                       (slice_img.max() - slice_img.min() + 1e-6) * 255).astype(np.uint8))
            img_pil = img_pil.rotate(angle, fillcolor=0)
            slice_img = np.array(img_pil) / 255.0
            mean, std = slice_img.mean(), slice_img.std()
            if std > 1e-6:
                slice_img = (slice_img - mean) / std
        
        if np.random.rand() > 0.5:
            img_min, img_max = slice_img.min(), slice_img.max()
            img_norm = (slice_img - img_min) / (img_max - img_min + 1e-6)
            gamma = np.random.uniform(0.8, 1.2)
            img_norm = np.power(img_norm, gamma)
            slice_img = img_norm * (img_max - img_min) + img_min
        
        if np.random.rand() > 0.5:
            noise = np.random.normal(0, 0.05, slice_img.shape)
            slice_img = slice_img + noise
        
        if np.random.rand() > 0.5:
            factor = np.random.uniform(0.9, 1.1)
            slice_img = slice_img * factor
        
        if np.random.rand() > 0.5:
            mean = slice_img.mean()
            slice_img = (slice_img - mean) * np.random.uniform(0.9, 1.1) + mean
        
        return slice_img
    
    def __len__(self):
        return len(self.df)
    
    def __getitem__(self, idx):
        try:
            row = self.df.iloc[idx]
            series_id = row['SeriesInstanceUID']
            
            if self.cache_dir and CFG.USE_CACHE:
                cache_path = self.get_cache_path(series_id)
                if os.path.exists(cache_path):
                    volume = np.load(cache_path)
                else:
                    volume = self.load_dicom_series(series_id)
            else:
                volume = self.load_dicom_series(series_id)
            
            if self.augment:
                volume = np.stack([self.augment_slice(s) for s in volume])
            
            labels = row[self.label_cols].values.astype(np.float32)
            
            if self.label_smoothing > 0:
                labels = labels * (1 - self.label_smoothing) + self.label_smoothing / 2
            
            volume = volume.astype(np.float32)
            return torch.from_numpy(volume).float(), torch.from_numpy(labels).float()
        except:
            return (
                torch.zeros(self.num_slices, self.image_size, self.image_size, dtype=torch.float32),
                torch.zeros(len(self.label_cols), dtype=torch.float32)
            )

print("Enhanced Dataset defined!")


# ============================================
# Cell 5: Enhanced Model with Attention Pooling
# ============================================
class PositionalEncoding(nn.Module):
    def __init__(self, d_model, max_len=25):
        super().__init__()
        pe = torch.zeros(max_len, d_model)
        position = torch.arange(0, max_len, dtype=torch.float).unsqueeze(1)
        div_term = torch.exp(torch.arange(0, d_model, 2).float() * (-np.log(10000.0) / d_model))
        pe[:, 0::2] = torch.sin(position * div_term)
        pe[:, 1::2] = torch.cos(position * div_term)
        self.register_buffer('pe', pe)
    
    def forward(self, x):
        return x + self.pe[:x.size(1), :]


class TransformerAneurysmModel(nn.Module):
    def __init__(self, backbone='vit_small_patch16_224', num_slices=20, num_classes=14,
                 embed_dim=384, num_heads=8, transformer_layers=4, dropout=0.5):
        super().__init__()
        
        self.backbone = timm.create_model(
            backbone, pretrained=True, in_chans=1, num_classes=0, global_pool='token'
        )
        
        total_params = sum(p.numel() for p in self.backbone.parameters())
        frozen_count = 0
        for name, param in self.backbone.named_parameters():
            if 'blocks.0' in name or 'blocks.1' in name:
                param.requires_grad = False
                frozen_count += param.numel()
        print(f"Frozen {frozen_count}/{total_params} backbone params ({frozen_count/total_params*100:.1f}%)")
        
        self.embed_dim = embed_dim
        self.pos_encoding = PositionalEncoding(embed_dim, max_len=num_slices)
        
        encoder_layer = nn.TransformerEncoderLayer(
            d_model=embed_dim,
            nhead=num_heads,
            dim_feedforward=embed_dim * 2,
            dropout=dropout,
            activation='gelu',
            batch_first=True
        )
        self.transformer = nn.TransformerEncoder(encoder_layer, num_layers=transformer_layers)
        
        self.attention_weights = nn.Linear(embed_dim, 1)
        
        self.classifier = nn.Sequential(
            nn.LayerNorm(embed_dim),
            nn.Dropout(dropout),
            nn.Linear(embed_dim, embed_dim // 4),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(embed_dim // 4, num_classes)
        )
    
    def forward(self, x):
        B, D, H, W = x.shape
        
        slice_features = []
        for i in range(D):
            feat = self.backbone(x[:, i:i+1, :, :])
            slice_features.append(feat)
        
        slice_features = torch.stack(slice_features, dim=1)
        slice_features = self.pos_encoding(slice_features)
        
        transformed = self.transformer(slice_features)
        
        attn_logits = self.attention_weights(transformed).squeeze(-1)
        attn_weights = F.softmax(attn_logits, dim=1).unsqueeze(-1)
        pooled = (attn_weights * transformed).sum(dim=1)
        
        output = self.classifier(pooled)
        return output

print("Model defined!")


# ============================================
# Cell 6: Helper Functions
# ============================================
def compute_auc(y_true, y_pred):
    aucs = []
    for i in range(y_true.shape[1]):
        try:
            if len(np.unique(y_true[:, i])) >= 2:
                auc = roc_auc_score(y_true[:, i], y_pred[:, i])
                aucs.append(auc)
        except:
            pass
    
    if len(aucs) == 0:
        return 0.0
    
    aneurysm_auc = aucs[-1] if len(aucs) == 14 else np.mean(aucs)
    other_auc = np.mean(aucs[:-1]) if len(aucs) == 14 else np.mean(aucs)
    
    return (aneurysm_auc + other_auc) / 2


def train_epoch(model, loader, criterion, optimizer, device, ema=None, accum_steps=1):
    model.train()
    losses = []
    ok, fail = 0, 0
    
    optimizer.zero_grad()
    
    for batch_idx, (images, labels) in enumerate(tqdm(loader, desc='Train')):
        try:
            images, labels = images.to(device), labels.to(device)
            
            outputs = model(images)
            loss = criterion(outputs, labels)
            loss = loss / accum_steps
            loss.backward()
            
            if (batch_idx + 1) % accum_steps == 0:
                torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
                optimizer.step()
                optimizer.zero_grad()
                
                if ema is not None:
                    ema.update()
            
            losses.append(loss.item() * accum_steps)
            ok += 1
        except Exception as e:
            fail += 1
            continue
    
    if (batch_idx + 1) % accum_steps != 0:
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
        optimizer.step()
        optimizer.zero_grad()
        if ema is not None:
            ema.update()
    
    print(f"Train - OK: {ok}, FAIL: {fail}")
    return np.mean(losses) if losses else 0.0


@torch.no_grad()
def valid_epoch(model, loader, criterion, device, ema=None):
    if ema is not None:
        ema.store()
        ema.copy_to()
    
    model.eval()
    losses, all_labels, all_preds = [], [], []
    ok, fail = 0, 0
    
    for images, labels in tqdm(loader, desc='Valid'):
        try:
            images, labels = images.to(device), labels.to(device)
            outputs = model(images)
            loss = criterion(outputs, labels)
            
            losses.append(loss.item())
            all_labels.append(labels.cpu().numpy())
            all_preds.append(torch.sigmoid(outputs).cpu().numpy())
            ok += 1
        except Exception as e:
            fail += 1
            continue
    
    if ema is not None:
        ema.restore()
    
    print(f"Valid - OK: {ok}, FAIL: {fail}")
    
    if not losses:
        return 0.0, 0.0
    
    all_labels = np.vstack(all_labels)
    all_preds = np.vstack(all_preds)
    auc = compute_auc(all_labels, all_preds)
    
    return np.mean(losses), auc


def plot_training_history(history, save_path='training_curves.png'):
    fig, axes = plt.subplots(1, 2, figsize=(15, 5))
    
    axes[0].plot(history['train_loss'], label='Train Loss', linewidth=2)
    axes[0].plot(history['valid_loss'], label='Valid Loss', linewidth=2)
    axes[0].set_xlabel('Epoch', fontsize=12)
    axes[0].set_ylabel('Loss', fontsize=12)
    axes[0].set_title('Training and Validation Loss', fontsize=14, fontweight='bold')
    axes[0].legend(fontsize=11)
    axes[0].grid(True, alpha=0.3)
    
    axes[1].plot(history['valid_auc'], label='Valid AUC', color='green', linewidth=2)
    axes[1].axhline(y=max(history['valid_auc']), color='r', linestyle='--', 
                    label=f'Best: {max(history["valid_auc"]):.4f}', linewidth=1.5)
    axes[1].set_xlabel('Epoch', fontsize=12)
    axes[1].set_ylabel('AUC', fontsize=12)
    axes[1].set_title('Validation AUC', fontsize=14, fontweight='bold')
    axes[1].legend(fontsize=11)
    axes[1].grid(True, alpha=0.3)
    
    plt.tight_layout()
    plt.savefig(save_path, dpi=150, bbox_inches='tight')
    plt.close()

print("Helper functions defined!")


# ============================================
# Cell 7: Single Model Training Function
# ============================================
def train_single_model(seed, train_df, valid_df):
    """Train a single model with given seed"""
    print(f"\n{'='*70}")
    print(f"TRAINING MODEL WITH SEED {seed}")
    print(f"{'='*70}")
    
    # Set seed
    torch.manual_seed(seed)
    np.random.seed(seed)
    
    # Create datasets
    train_dataset = RSNADataset(
        train_df, CFG.DATA_DIR, CFG.NUM_SLICES, CFG.IMAGE_SIZE, 
        augment=True, cache_dir=CFG.CACHE_DIR, label_smoothing=CFG.LABEL_SMOOTHING
    )
    valid_dataset = RSNADataset(
        valid_df, CFG.DATA_DIR, CFG.NUM_SLICES, CFG.IMAGE_SIZE, 
        augment=False, cache_dir=CFG.CACHE_DIR, label_smoothing=0.0
    )
    
    train_loader = DataLoader(
        train_dataset, batch_size=CFG.BATCH_SIZE, shuffle=True, 
        num_workers=CFG.NUM_WORKERS, pin_memory=True
    )
    valid_loader = DataLoader(
        valid_dataset, batch_size=CFG.BATCH_SIZE, shuffle=False, 
        num_workers=CFG.NUM_WORKERS, pin_memory=True
    )
    
    # Create model
    model = TransformerAneurysmModel(
        backbone=CFG.BACKBONE,
        num_slices=CFG.NUM_SLICES,
        num_classes=CFG.NUM_CLASSES,
        embed_dim=CFG.EMBED_DIM,
        num_heads=CFG.NUM_HEADS,
        transformer_layers=CFG.TRANSFORMER_LAYERS,
        dropout=CFG.DROPOUT
    )
    model.to(CFG.DEVICE)
    
    criterion = FocalLoss(alpha=0.25, gamma=2.0)
    optimizer = optim.AdamW(model.parameters(), lr=CFG.LR, weight_decay=2e-4, betas=(0.9, 0.999))
    
    from torch.optim.lr_scheduler import CosineAnnealingWarmRestarts
    scheduler = CosineAnnealingWarmRestarts(optimizer, T_0=5, T_mult=2, eta_min=1e-6)
    
    ema = ExponentialMovingAverage(model.parameters(), decay=0.995) if CFG.USE_EMA else None
    
    # Training loop
    best_auc = 0
    patience_counter = 0
    history = {'train_loss': [], 'valid_loss': [], 'valid_auc': []}
    
    for epoch in range(CFG.EPOCHS):
        print(f"\nEpoch {epoch+1}/{CFG.EPOCHS} | LR: {optimizer.param_groups[0]['lr']:.2e}")
        start_time = time.time()
        
        train_loss = train_epoch(
            model, train_loader, criterion, optimizer, CFG.DEVICE, 
            ema=ema, accum_steps=CFG.GRADIENT_ACCUMULATION
        )
        valid_loss, valid_auc = valid_epoch(model, valid_loader, criterion, CFG.DEVICE, ema=ema)
        scheduler.step()
        
        history['train_loss'].append(train_loss)
        history['valid_loss'].append(valid_loss)
        history['valid_auc'].append(valid_auc)
        
        epoch_time = time.time() - start_time
        print(f"Time: {epoch_time/60:.1f}min | Train: {train_loss:.4f} | Valid: {valid_loss:.4f}, AUC: {valid_auc:.4f}")
        
        if valid_auc > best_auc:
            best_auc = valid_auc
            patience_counter = 0
            
            save_dict = {
                'epoch': epoch,
                'model_state_dict': model.state_dict(),
                'auc': valid_auc,
                'seed': seed
            }
            if ema:
                ema.store()
                ema.copy_to()
                save_dict['ema_model_state_dict'] = model.state_dict()
                ema.restore()
            
            torch.save(save_dict, f'best_model_seed{seed}.pth')
            print(f"Best model saved! AUC: {valid_auc:.4f}")
        else:
            patience_counter += 1
            print(f"No improvement ({patience_counter}/{CFG.PATIENCE})")
        
        if patience_counter >= CFG.PATIENCE:
            print(f"\nEarly stopping at epoch {epoch+1}")
            break
    
    # Save history
    with open(f'history_seed{seed}.pkl', 'wb') as f:
        pickle.dump(history, f)
    
    # Plot
    plot_training_history(history, f'curves_seed{seed}.png')
    
    print(f"\nSeed {seed} complete | Best AUC: {best_auc:.4f}")
    
    return best_auc, history

print("Training function defined!")


# ============================================
# Cell 8: Model Training
# ============================================
print("\n" + "="*70)
print("STARTING MODEL TRAINING")
print("="*70)
print(f"Seed: {CFG.SEEDS[0]}")
print("="*70)

# Load and split data
df = pd.read_csv(CFG.TRAIN_CSV)
print(f"\nTotal samples: {len(df)}")

if CFG.USE_SUBSET:
    df = df.sample(n=min(CFG.USE_SUBSET, len(df)), random_state=42)

train_df, valid_df = train_test_split(
    df, test_size=0.15, random_state=42, stratify=df['Aneurysm Present']
)
print(f"Train: {len(train_df)}, Valid: {len(valid_df)}")

# Train model
seed = CFG.SEEDS[0]
best_auc, history = train_single_model(seed, train_df, valid_df)

print("\n" + "="*70)
print("TRAINING COMPLETE")
print("="*70)
print(f"Best AUC: {best_auc:.4f}")


# ============================================
# Cell 9: Model Validation
# ============================================
print("\n" + "="*70)
print("MODEL VALIDATION")
print("="*70)

# Load trained model
seed = CFG.SEEDS[0]
model = TransformerAneurysmModel(
    backbone=CFG.BACKBONE,
    num_slices=CFG.NUM_SLICES,
    num_classes=CFG.NUM_CLASSES,
    embed_dim=CFG.EMBED_DIM,
    num_heads=CFG.NUM_HEADS,
    transformer_layers=CFG.TRANSFORMER_LAYERS,
    dropout=CFG.DROPOUT
).to(CFG.DEVICE)

checkpoint = torch.load(f'best_model_seed{seed}.pth', weights_only=False)
if 'ema_model_state_dict' in checkpoint:
    model.load_state_dict(checkpoint['ema_model_state_dict'])
else:
    model.load_state_dict(checkpoint['model_state_dict'])

model.eval()
print(f"Loaded model seed {seed} (AUC: {checkpoint['auc']:.4f})")


@torch.no_grad()
def model_predict(model, loader, device):
    """Model prediction"""
    all_labels = []
    all_preds = []
    
    for images, labels in tqdm(loader, desc='Predict'):
        images = images.to(device)
        
        outputs = model(images)
        probs = torch.sigmoid(outputs)
        
        all_labels.append(labels.cpu().numpy())
        all_preds.append(probs.cpu().numpy())
    
    all_labels = np.vstack(all_labels)
    all_preds = np.vstack(all_preds)
    
    return all_labels, all_preds


# Create validation loader
valid_dataset = RSNADataset(
    valid_df, CFG.DATA_DIR, CFG.NUM_SLICES, CFG.IMAGE_SIZE, 
    augment=False, cache_dir=CFG.CACHE_DIR
)
valid_loader = DataLoader(
    valid_dataset, batch_size=CFG.BATCH_SIZE, shuffle=False, 
    num_workers=CFG.NUM_WORKERS, pin_memory=True
)

# Get predictions
y_true, y_pred = model_predict(model, valid_loader, CFG.DEVICE)
final_auc = compute_auc(y_true, y_pred)

print(f"\nFINAL RESULTS:")
print(f"{'='*70}")
print(f"Validation AUC: {final_auc:.4f}")
print(f"{'='*70}")


# ============================================
# Cell 10: Visualization
# ============================================
print("\nCreating visualization...")

fig, axes = plt.subplots(1, 2, figsize=(15, 5))

# Plot 1: Training curves
axes[0].plot(history['train_loss'], label='Train Loss', linewidth=2)
axes[0].plot(history['valid_loss'], label='Valid Loss', linewidth=2)
axes[0].set_xlabel('Epoch', fontsize=12)
axes[0].set_ylabel('Loss', fontsize=12)
axes[0].set_title('Training and Validation Loss', fontsize=14, fontweight='bold')
axes[0].legend(fontsize=11)
axes[0].grid(True, alpha=0.3)

# Plot 2: AUC curve
axes[1].plot(history['valid_auc'], label='Valid AUC', color='green', linewidth=2)
axes[1].axhline(y=best_auc, color='r', linestyle='--', 
                label=f'Best: {best_auc:.4f}', linewidth=1.5)
axes[1].set_xlabel('Epoch', fontsize=12)
axes[1].set_ylabel('AUC', fontsize=12)
axes[1].set_title('Validation AUC', fontsize=14, fontweight='bold')
axes[1].legend(fontsize=11)
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('training_analysis.png', dpi=150, bbox_inches='tight')
print("Visualization saved to training_analysis.png")
plt.show()


# ============================================
# Cell 11: Save Predictions
# ============================================
print("\nSaving predictions...")

# Save predictions
predictions_df = pd.DataFrame({
    'predictions': y_pred.tolist(),
    'labels': y_true.tolist()
})
predictions_df.to_pickle('predictions.pkl')
print("Predictions saved to predictions.pkl")

# Save metadata
metadata = {
    'seed': seed,
    'best_auc': best_auc,
    'final_auc': final_auc,
    'config': {
        'num_slices': CFG.NUM_SLICES,
        'backbone': CFG.BACKBONE,
        'transformer_layers': CFG.TRANSFORMER_LAYERS,
        'num_heads': CFG.NUM_HEADS,
    }
}
with open('metadata.pkl', 'wb') as f:
    pickle.dump(metadata, f)
print("Metadata saved")


# ============================================
# Cell 12: Inference Test
# ============================================
print("\nTesting inference...")

with torch.no_grad():
    dummy = torch.randn(1, CFG.NUM_SLICES, CFG.IMAGE_SIZE, CFG.IMAGE_SIZE).to(CFG.DEVICE)
    output = torch.sigmoid(model(dummy))
    
    print(f"Inference test passed!")
    print(f"  Output shape: {output.shape}")
    print(f"  Sample predictions: {output[0][:3].cpu().numpy()}")

print("\n" + "="*70)
print("TRAINING PIPELINE COMPLETE!")
print("="*70)
print(f"\nSaved files:")
print(f"  - best_model_seed{seed}.pth")
print(f"  - predictions.pkl")
print(f"  - metadata.pkl")
print(f"  - training_analysis.png")
print("\nReady for submission or further analysis!")